# Base ICDAS v4 — re-corte a escala nativa uniforme
**Settings:** CPU, Internet OFF. **Inputs:** salida de `base-icdas-v3` (`icdas_v3.zip`, `manifest_privado_v3.csv`) y dataset de Ahmed completo (`Dataset/Images`).

Motivo: en v3 los recortes de la base original estaban guardados a ~1:2 de la foto y los minados a escala nativa; el diagnóstico de resolución mostró que el clasificador usaba esa diferencia como atajo. En v4 **todos** los recortes se cortan de nuevo desde la foto original con la misma caja, a escala nativa (sin re-escalar), igual que los recortes automáticos del pipeline. Mismos IDs, etiquetas y folds que v3.

In [ ]:
import os, glob, zipfile, shutil, hashlib, numpy as np, pandas as pd, cv2
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
find = lambda n: (glob.glob(f'{IN}/**/{n}', recursive=True) or [None])[0]
IMGDIR = [d for d in glob.glob(f'{IN}/**/Dataset/Images', recursive=True) if os.path.isdir(d)][0]
zipfile.ZipFile(find('icdas_v3.zip')).extractall(f'{TMP}/v3')
V3 = f'{TMP}/v3/icdas_v3'
man = pd.read_csv(f'{V3}/manifest.csv', dtype={'micdas': str})
priv = pd.read_csv(find('manifest_privado_v3.csv'))
print(len(man), 'recortes'); print(man.micdas.value_counts())
OUT = f'{W}/icdas_v4'; os.makedirs(f'{OUT}/crops', exist_ok=True)
H, Wd, md5, ok, same = [], [], [], [], []
for r in man.itertuples():
    im = cv2.imread(os.path.join(IMGDIR, r.ahmed_image))
    assert im is not None, r.ahmed_image
    x0, y0 = max(0, int(r.x0)), max(0, int(r.y0)); x1, y1 = min(im.shape[1], int(r.x1)), min(im.shape[0], int(r.y1))
    c = im[y0:y1, x0:x1]
    assert c.size and min(c.shape[:2]) >= 16, r.crop_id
    f = f'{OUT}/crops/{r.crop_id}.png'; cv2.imwrite(f, c)
    H.append(c.shape[0]); Wd.append(c.shape[1]); md5.append(hashlib.md5(open(f, 'rb').read()).hexdigest())
    old = cv2.imread(f'{V3}/{r.crop_file}')
    same.append(old.shape == c.shape and np.abs(old.astype(int) - c.astype(int)).mean() < 1.0)
man['escala_v3'] = man.w / (man.x1 - man.x0)
man['h'], man['w'], man['md5'] = H, Wd, md5
man['igual_a_v3'] = same
o = man.crop_id.map(priv.set_index('crop_id_v3').origen)
print('idénticos a v3 por origen (los minados deberían serlo):'); print(pd.crosstab(o, man.igual_a_v3))
print('escala v3 por origen:'); print(man.groupby(o).escala_v3.median())
print(man[['h', 'w']].describe().round(0).T)

In [ ]:
pub = man.drop(columns=['md5', 'igual_a_v3', 'escala_v3'])
pub.to_csv(f'{OUT}/manifest.csv', index=False)
readme = open(f'{V3}/README.md').read().replace('(v3)', '(v4)')
readme += '\n- v4: every crop re-cut from the original photograph with the same box at native resolution (no rescaling), so all crops share one scale.\n'
open(f'{OUT}/README.md', 'w').write(readme)
p4 = priv.copy(); p4['md5_v4'] = p4.crop_id_v3.map(man.set_index('crop_id').md5)
p4.to_csv(f'{W}/manifest_privado_v4.csv', index=False)
shutil.make_archive(f'{W}/icdas_v4', 'zip', W, 'icdas_v4'); shutil.rmtree(OUT)
print({f: round(os.path.getsize(f'{W}/{f}') / 1e6, 1) for f in os.listdir(W)})